In [1]:
import os
import math
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import rcParams
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "SimSun", "Arial Unicode MS"]
rcParams["axes.unicode_minus"] = False
# =========================
# 0) 配置
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_DEF_PY = r"C:\Users\Administrator\Downloads\1.py"

# .pt或者.pth文件
MODEL_PATH = r"C:\Users\Administrator\Downloads\BIBLIS.pth"

# 参考解点集 CSV
CSV_PATH   = r"C:\Users\Administrator\Downloads\BIBLIS.csv"

CSV_USE_COLNAMES = False
CSV_COLS_BY_INDEX = dict(x=0, y=1, phi1=3, phi2=4)
CSV_COLS_BY_NAME  = dict(x="x", y="y", phi1="phi1", phi2="phi2")


NU_SIGMA_F1 = {1: 0.005871, 2: 0.006191, 3: 0.0, 4: 0.007453, 5: 0.006191, 6: 0.006429, 7: 0.006191, 8: 0.006429}
NU_SIGMA_F2 = {1: 0.096067, 2: 0.103580, 3: 0.0, 4: 0.132360, 5: 0.103580, 6: 0.109110, 7: 0.103580, 8: 0.109110}

# 分母保护（相对误差）
REL_EPS_FACTOR = 0.1

# 可视化裁剪（避免极端点把色标撑爆；None 表示不裁剪）
REL_ERR_CLIP = 1.0     # 带符号相对误差裁剪到 [-REL_ERR_CLIP, +REL_ERR_CLIP]；可设 None
ABS_ERR_CLIP = None    # 带符号绝对误差裁剪到 [-ABS_ERR_CLIP, +ABS_ERR_CLIP]；可设 None

# 是否保存图像（评估脚本默认会 plt.show()；保存则额外写入 OUT_DIR）
SAVE_FIG = True
OUT_DIR = "./eval_plots"


# =========================
# 1) 训练网络定义动态导入（避免手动复制网络结构）
# =========================
def load_train_module(py_path: str):
    py_path = str(Path(py_path).resolve())
    spec = importlib.util.spec_from_file_location("train_model_def", py_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"无法从 {py_path} 导入模块")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

# =========================
# 2) 评估用材料判定（模仿训练：边界点往域内挪一点）
# =========================
L = 196.5421
EXT_SEGMENTS = [
    (0, L, L, L, 0, 1),    # 顶边 y=L
    (L, L, 0, L, 1, 0),    # 右边 x=L
]

def _on_segment_mask(x, y, x1, x2, y1, y2, tol):
    if y1 == y2:  # 水平
        ymask = torch.isclose(y, torch.full_like(y, float(y1)), atol=tol)
        xmin, xmax = (min(x1,x2), max(x1,x2))
        xmask = (x >= xmin - tol) & (x <= xmax + tol)
        return ymask & xmask
    else:         # 垂直
        xmask = torch.isclose(x, torch.full_like(x, float(x1)), atol=tol)
        ymin, ymax = (min(y1,y2), max(y1,y2))
        ymask = (y >= ymin - tol) & (y <= ymax + tol)
        return xmask & ymask

def get_material_id_eval(get_material_id_fn, x_coords, y_coords, eps=1e-3, tol=1e-6, fallback_eps=1e-3):

    x = x_coords if x_coords.dim() == 2 else x_coords.view(-1, 1)
    y = y_coords if y_coords.dim() == 2 else y_coords.view(-1, 1)
    x_in = x.clone()
    y_in = y.clone()

    # Mirror 边界（x=0,y=0）：往域内挪
    on_x0 = torch.isclose(x, torch.zeros_like(x), atol=tol)
    x_in[on_x0] = x[on_x0] + eps
    on_y0 = torch.isclose(y, torch.zeros_like(y), atol=tol)
    y_in[on_y0] = y[on_y0] + eps

    # EXT_NULL 台阶外边界：按训练 EXT_SEGMENTS 往域内挪
    for x1,x2,y1,y2,nx,ny in EXT_SEGMENTS:
        m = _on_segment_mask(x, y, x1, x2, y1, y2, tol)
        if m.any():
            x_in[m] = x[m] - eps * float(nx)
            y_in[m] = y[m] - eps * float(ny)

    mat = get_material_id_fn(x_in, y_in)

    zero = (mat == 0)
    if zero.any():
        offsets = [
            (+fallback_eps, 0.0), (-fallback_eps, 0.0),
            (0.0, +fallback_eps), (0.0, -fallback_eps),
            (+fallback_eps, +fallback_eps), (+fallback_eps, -fallback_eps),
            (-fallback_eps, +fallback_eps), (-fallback_eps, -fallback_eps),
        ]
        mat2 = mat.clone()
        for dx, dy in offsets:
            cand = get_material_id_fn(x_in + dx, y_in + dy)
            pick = (mat2 == 0) & (cand != 0)
            mat2[pick] = cand[pick]
        mat = mat2

    return mat.squeeze()

# =========================
# 3) 误差指标
# =========================
def compute_metrics(ref: np.ndarray, pred: np.ndarray, rel_eps: float):
    ref = np.asarray(ref).reshape(-1)
    pred = np.asarray(pred).reshape(-1)
    diff = pred - ref
    mse = float(np.mean(diff**2))

    rmse = float(math.sqrt(mse))
    mae = float(np.mean(np.abs(diff)))
    l2_num = float(np.linalg.norm(diff, ord=2))
    l2_den = float(np.linalg.norm(ref, ord=2) + 1e-30)
    l2_rel = l2_num / l2_den
    rel = np.abs(diff) / (np.abs(ref) + rel_eps)
    max_rel = float(np.max(rel)) if rel.size else float("nan")
    mean_rel = float(np.mean(rel)) if rel.size else float("nan")
    sse = float(np.sum(diff**2))
    ref_mean = float(np.mean(ref)) if ref.size else 0.0
    sst = float(np.sum((ref - ref_mean)**2))
    r2 = float("nan") if sst < 1e-30 else (1.0 - sse / (sst + 1e-30))
    return {"L2_rel": l2_rel, "MSE": mse, "RMSE": rmse, "MAE": mae,
            "MaxRelErr": max_rel, "MeanRelErr": mean_rel, "R2": r2}

def region_masks(mat_ids_np: np.ndarray):
    masks = {}
    for i in range(1, 9):
        masks[f"mat{i}"] = (mat_ids_np == i)
    
    # BIBLIS 中 3 号是 Reflector，其余是 Fuel
    masks["fuel"] = np.isin(mat_ids_np, [1, 2, 4, 5, 6, 7, 8])
    masks["reflector"] = (mat_ids_np == 3)
    masks["all"] = np.isin(mat_ids_np, list(range(1, 9)))
    return masks

def nu_sigma_f_from_mat(mat_ids_np: np.ndarray):
    nu1 = np.zeros_like(mat_ids_np, dtype=np.float32)
    nu2 = np.zeros_like(mat_ids_np, dtype=np.float32)
    for k in range(1, 9):
        nu1[mat_ids_np == k] = NU_SIGMA_F1[k]
        nu2[mat_ids_np == k] = NU_SIGMA_F2[k]
    return nu1, nu2
# =========================
# 5) 对角线剖面图（训练后评估）
# =========================
def make_line_points(p0, p1, n=600):
    x0, y0 = p0
    x1, y1 = p1
    t = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x = x0 + (x1 - x0) * t
    y = y0 + (y1 - y0) * t
    s = np.sqrt((x - x0) ** 2 + (y - y0) ** 2).astype(np.float32)  # 沿线弧长坐标
    return x, y, s

def idw_interp_scattered(x_ref, y_ref, v_ref, xq, yq, k=8, power=2.0, chunk=256):

    pts = np.stack([x_ref, y_ref], axis=1).astype(np.float64)
    vals = np.asarray(v_ref, dtype=np.float64).reshape(-1)
    q = np.stack([xq, yq], axis=1).astype(np.float64)

    n_ref = pts.shape[0]
    k = max(1, min(int(k), n_ref))
    out = np.empty(len(q), dtype=np.float64)

    for i in range(0, len(q), chunk):
        qq = q[i:i+chunk]  # [m,2]
        d2 = ((qq[:, None, 0] - pts[None, :, 0]) ** 2 +
              (qq[:, None, 1] - pts[None, :, 1]) ** 2)  # [m,n_ref]

        idx = np.argpartition(d2, kth=k-1, axis=1)[:, :k]          # [m,k]
        d2_k = np.take_along_axis(d2, idx, axis=1)                 # [m,k]
        v_k = vals[idx]                                            # [m,k]

        # 若恰好命中散点，直接取该值
        exact = d2_k <= 1e-24
        w = 1.0 / np.maximum(d2_k, 1e-24) ** (power / 2.0)
        out_chunk = np.sum(w * v_k, axis=1) / np.sum(w, axis=1)

        rows = np.where(np.any(exact, axis=1))[0]
        for r in rows:
            j = np.argmin(d2_k[r])
            out_chunk[r] = v_k[r, j]

        out[i:i+chunk] = out_chunk

    return out.astype(np.float32)

@torch.no_grad()
def predict_profile_line(mod, model, p0, p1, device, n=600):
    x_line, y_line, s_line = make_line_points(p0, p1, n=n)

    xy_t = torch.from_numpy(np.stack([x_line, y_line], axis=1).astype(np.float32)).to(device)
    x_t = xy_t[:, 0:1]
    y_t = xy_t[:, 1:2]

    mat_ids_t = get_material_id_eval(
        mod.get_material_id, x_t, y_t,
        eps=1e-3, tol=1e-6, fallback_eps=1e-3
    ).long().view(-1)

    mat_ids = mat_ids_t.cpu().numpy().astype(np.int64)
    valid = mat_ids > 0

    phi1_pred = np.full(len(x_line), np.nan, dtype=np.float32)
    phi2_pred = np.full(len(x_line), np.nan, dtype=np.float32)

    if np.any(valid):
        phi1_t, phi2_t,*_ = model(xy_t[valid], mat_ids_t[valid])
        phi1_pred[valid] = phi1_t.squeeze(-1).cpu().numpy().astype(np.float32)
        phi2_pred[valid] = phi2_t.squeeze(-1).cpu().numpy().astype(np.float32)

    return x_line, y_line, s_line, mat_ids, phi1_pred, phi2_pred

def find_interface_positions_on_line(s_line, mat_ids):

    mats = np.asarray(mat_ids).reshape(-1)
    valid_pair = (mats[:-1] > 0) & (mats[1:] > 0)
    jump = valid_pair & (mats[:-1] != mats[1:])
    idx = np.where(jump)[0]
    return [0.5 * (s_line[i] + s_line[i+1]) for i in idx]

def plot_diagonal_profile(
    mod, model,
    x_ref_all, y_ref_all, phi1_ref_all, phi2_ref_all,
    p0, p1,
    S_ref, scale,
    case_name="BIBLIS",
    save_fig=True, out_dir="./eval_plots",
    n_line=600
):
    # 1) 模型沿线预测（原始输出）
    x_line, y_line, s_line, mat_line, phi1_pred, phi2_pred = predict_profile_line(
        mod, model, p0, p1, DEVICE, n=n_line
    )

    # 2) 参考解沿线插值（原始参考）
    phi1_ref_line = idw_interp_scattered(
        x_ref_all, y_ref_all, phi1_ref_all, x_line, y_line, k=8, power=2.0
    )
    phi2_ref_line = idw_interp_scattered(
        x_ref_all, y_ref_all, phi2_ref_all, x_line, y_line, k=8, power=2.0
    )

    # 3) 与主评估一致：统一裂变源归一化
    eps = 1e-30
    phi1_ref_line = phi1_ref_line / (S_ref + eps)
    phi2_ref_line = phi2_ref_line / (S_ref + eps)

    phi1_pred = phi1_pred * scale / (S_ref + eps)
    phi2_pred = phi2_pred * scale / (S_ref + eps)

    # 4) 域外点置 NaN
    valid = mat_line > 0
    phi1_ref_line[~valid] = np.nan
    phi2_ref_line[~valid] = np.nan
    phi1_pred[~valid] = np.nan
    phi2_pred[~valid] = np.nan

    # 5) 材料界面位置
    interface_s = find_interface_positions_on_line(s_line, mat_line)

    # 6) 画图
    fig, axes = plt.subplots(2, 1, figsize=(8.5, 6.8), sharex=True)

    axes[0].plot(s_line, phi1_ref_line, label="Reference", linewidth=2.0)
    axes[0].plot(s_line, phi1_pred, label="Prediction", linewidth=1.6)
    axes[0].set_ylabel(r"$\phi_1$")
    axes[0].set_title(f"{case_name} diagonal profile (group 1)")

    axes[1].plot(s_line, phi2_ref_line, label="Reference", linewidth=2.0)
    axes[1].plot(s_line, phi2_pred, label="Prediction", linewidth=1.6)
    axes[1].set_ylabel(r"$\phi_2$")
    axes[1].set_xlabel("Distance along the diagonal (cm)")
    axes[1].set_title(f"{case_name} diagonal profile (group 2)")

    for ax in axes:
        for ss in interface_s:
            ax.axvline(ss, linestyle="--", linewidth=1.0, alpha=0.6)
        ax.grid(True, alpha=0.25)
        ax.legend()

    plt.tight_layout()

    if save_fig:
        os.makedirs(out_dir, exist_ok=True)
        fname = f"{case_name.lower()}_diagonal_profile.png"
        plt.savefig(os.path.join(out_dir, fname), dpi=300, bbox_inches="tight")

    plt.show()
# =========================
# 4) 主流程
# =========================
def main():
    import pandas as pd
    import numpy as np
    import torch

    # ---- 4.1 导入训练脚本模块（确保网络结构一致）----
    mod = load_train_module(MODEL_DEF_PY)

    if hasattr(mod, "build_true_interface_lines"):
        mod.TRUE_X_IF, mod.TRUE_Y_IF = mod.build_true_interface_lines(eps=1e-2, n_scan=512, device=DEVICE)

    # ---- 4.2 实例化模型并加载权重 ----
    model = mod.MultiMatPINN().to(DEVICE)

    obj = torch.load(MODEL_PATH, map_location=DEVICE)
    state_dict = obj["model"] if isinstance(obj, dict) and "model" in obj else obj
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        print("[提示] load_state_dict 非严格匹配：")
        if missing: print("  missing keys:", missing[:10], "..." if len(missing)>10 else "")
        if unexpected: print("  unexpected keys:", unexpected[:10], "..." if len(unexpected)>10 else "")
    model.eval()
    print("模型加载完成。")

    # ---- 4.3 读取 CSV ----
    df = pd.read_csv(CSV_PATH)
    if CSV_USE_COLNAMES:
        x = df[CSV_COLS_BY_NAME["x"]].to_numpy(np.float32)
        y = df[CSV_COLS_BY_NAME["y"]].to_numpy(np.float32)
        phi1_ref = df[CSV_COLS_BY_NAME["phi1"]].to_numpy(np.float32)
        phi2_ref = df[CSV_COLS_BY_NAME["phi2"]].to_numpy(np.float32)
    else:
        x = df.iloc[:, CSV_COLS_BY_INDEX["x"]].to_numpy(np.float32)
        y = df.iloc[:, CSV_COLS_BY_INDEX["y"]].to_numpy(np.float32)
        phi1_ref = df.iloc[:, CSV_COLS_BY_INDEX["phi1"]].to_numpy(np.float32)
        phi2_ref = df.iloc[:, CSV_COLS_BY_INDEX["phi2"]].to_numpy(np.float32)
    print(f"CSV 读取完成，共 {len(x)} 个点。")

    # ---- 4.4 模型预测 ----
    xy = np.stack([x, y], axis=1).astype(np.float32)
    xy_t = torch.from_numpy(xy).to(DEVICE)

    with torch.no_grad():
        x_t = xy_t[:, 0:1]
        y_t = xy_t[:, 1:2]
        mat_ids_t = get_material_id_eval(mod.get_material_id, x_t, y_t, eps=1e-3, tol=1e-6, fallback_eps=1e-3).long().view(-1)
        phi1_pred_t, phi2_pred_t,*_ = model(xy_t, mat_ids_t)

    phi1_pred = phi1_pred_t.squeeze(-1).cpu().numpy().astype(np.float32)
    phi2_pred = phi2_pred_t.squeeze(-1).cpu().numpy().astype(np.float32)
    mat_ids_np = mat_ids_t.cpu().numpy().astype(np.int64)

    masks = region_masks(mat_ids_np)
    in_domain = masks["all"]

    # ---- 4.5 裂变源归一化（让 pred/ref 在同一归一化下比较）----
    nu1, nu2 = nu_sigma_f_from_mat(mat_ids_np)
    S_ref = float(np.sum(nu1[in_domain] * phi1_ref[in_domain] + nu2[in_domain] * phi2_ref[in_domain]))
    S_pred = float(np.sum(nu1[in_domain] * phi1_pred[in_domain] + nu2[in_domain] * phi2_pred[in_domain]))
    eps = 1e-30
    if abs(S_ref) < 1e-20:
        raise RuntimeError("S_ref 过小（几乎为0），请检查CSV是否包含燃料区点/phi2列是否正确。")

    scale = S_ref / (S_pred + eps)
    phi1_pred_scaled = phi1_pred * scale
    phi2_pred_scaled = phi2_pred * scale

    phi1_ref_n  = phi1_ref / (S_ref + eps)
    phi2_ref_n  = phi2_ref / (S_ref + eps)
    phi1_pred_n = phi1_pred_scaled / (S_ref + eps)
    phi2_pred_n = phi2_pred_scaled / (S_ref + eps)

    print(f"裂变源：S_ref={S_ref:.6e}, S_pred={S_pred:.6e}, scale=S_ref/S_pred={scale:.6e}")

    rel_eps_1 = REL_EPS_FACTOR * float(np.max(np.abs(phi1_ref_n[in_domain]))) + 1e-30
    rel_eps_2 = REL_EPS_FACTOR * float(np.max(np.abs(phi2_ref_n[in_domain]))) + 1e-30
        # ---- 4.X 对角线剖面图：BIBLIS 对称对角线 (0,Ly) -> (Lx,0) ----
    Lx = 196.5421
    Ly = 196.5421

    plot_diagonal_profile(
        mod, model,
        x_ref_all=x,
        y_ref_all=y,
        phi1_ref_all=phi1_ref,
        phi2_ref_all=phi2_ref,
        p0=(0.0, Ly),
        p1=(Lx, 0.0),
        S_ref=S_ref,
        scale=scale,
        case_name="BIBLIS",
        save_fig=SAVE_FIG,
        out_dir=OUT_DIR,
        n_line=700
    )

    # ---- 4.6 逐区域统计 ----
    regions = ["mat1","mat2","mat3","mat4","fuel","reflector","all"]
    rows = []
    for reg in regions:
        m = masks[reg] & in_domain
        npts = int(np.sum(m))
        if npts == 0:
            metric_cols = ["L2_rel","MSE","RMSE","MAE","MaxRelErr","MeanRelErr","R2"]
            rows.append(dict(region=reg, N=0, group="phi1", **{k: np.nan for k in metric_cols}))
            rows.append(dict(region=reg, N=0, group="phi2", **{k: np.nan for k in metric_cols}))
            continue

        met1 = compute_metrics(phi1_ref_n[m], phi1_pred_n[m], rel_eps=rel_eps_1)
        met2 = compute_metrics(phi2_ref_n[m], phi2_pred_n[m], rel_eps=rel_eps_2)
        rows.append(dict(region=reg, N=npts, group="phi1", **met1))
        rows.append(dict(region=reg, N=npts, group="phi2", **met2))

    import pandas as pd
    result = pd.DataFrame(rows)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 50)
    print("\n===== 归一化后误差指标（按区域）=====")
    print(result.to_string(index=False, float_format=lambda v: f"{v:.6e}" if isinstance(v,float) and np.isfinite(v) else str(v)))


    # ---- 4.7 可视化：全域偏差分布图----

    try:
        import matplotlib.pyplot as plt
        import os as _os
        _os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
        err1 = (phi1_pred_n - phi1_ref_n)
        err2 = (phi2_pred_n - phi2_ref_n)
        rel_err1 = err1 / (np.abs(phi1_ref_n) + rel_eps_1)
        rel_err2 = err2 / (np.abs(phi2_ref_n) + rel_eps_2)

        def clip_sym(arr, clip_val):
            if clip_val is None:
                return arr, None
            return np.clip(arr, -clip_val, clip_val), clip_val

        err1_p, _   = clip_sym(err1, ABS_ERR_CLIP)
        err2_p, _   = clip_sym(err2, ABS_ERR_CLIP)
        mplot = in_domain
        
        # --- 新增：强制让绝对误差的 vmin 和 vmax 围绕 0 对称 ---
        vmax_err1 = np.max(np.abs(err1_p[mplot]))
        vmin_err1 = -vmax_err1
        vmax_err2 = np.max(np.abs(err2_p[mplot]))
        vmin_err2 = -vmax_err2
        
        rel1_p, v1  = clip_sym(rel_err1, REL_ERR_CLIP)
        rel2_p, v2  = clip_sym(rel_err2, REL_ERR_CLIP)

        mplot = in_domain
        x_p, y_p = x[mplot], y[mplot]

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))

        ax = axes[0, 0]
        sc = ax.scatter(x_p, y_p, c=err1_p[mplot], s=10, marker="s", cmap="seismic",
                        vmin=vmin_err1, vmax=vmax_err1)
        ax.set_title("归一化后 φ1 误差")
        ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("y")
        plt.colorbar(sc, ax=ax)

        ax = axes[0, 1]
        sc = ax.scatter(x_p, y_p, c=rel1_p[mplot], s=10, marker="s", cmap="seismic",
                        vmin=(-v1 if v1 is not None else None),
                        vmax=( v1 if v1 is not None else None))
        ax.set_title("φ1 的相对误差")
        ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("y")
        plt.colorbar(sc, ax=ax)

        ax = axes[1, 0]
        sc = ax.scatter(x_p, y_p, c=err2_p[mplot], s=10, marker="s", cmap="seismic",
                        vmin=vmin_err2, vmax=vmax_err2)
        ax.set_title("归一化后 φ2 误差")
        ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("y")
        plt.colorbar(sc, ax=ax)

        ax = axes[1, 1]
        sc = ax.scatter(x_p, y_p, c=rel2_p[mplot], s=10, marker="s", cmap="seismic",
                        vmin=(-v2 if v2 is not None else None),
                        vmax=( v2 if v2 is not None else None))
        ax.set_title("φ2 的相对误差")
        ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("y")
        plt.colorbar(sc, ax=ax)

        plt.tight_layout()
        if SAVE_FIG:
            _os.makedirs(OUT_DIR, exist_ok=True)
            out_path = _os.path.join(OUT_DIR, "global_error_maps1.png")
            plt.savefig(out_path, dpi=200)
            print(f"[已保存] 全域偏差分布图：{out_path}")

        plt.show()

    except Exception as e:
        print(f"[提示] 画全域偏差分布图时出错（不影响数值统计）：{e}")

if __name__ == "__main__":
    
    main()

[Run] OUTPUT_DIR = C:\Users\Administrator\Desktop\多尺度特征融合PINN代码文件夹\run_20260503-101805-pid37556
[Run] LOG_PATH   = C:\Users\Administrator\Desktop\多尺度特征融合PINN代码文件夹\run_20260503-101805-pid37556\train.log
[Run] START_TIME = 2026-05-03 10:18:05
TRUE_X_IF: [11.561300277709961, 34.68389892578125, 57.80649948120117, 80.9291000366211, 104.05169677734375, 127.17430114746094, 150.29690551757812, 173.41949462890625]
TRUE_Y_IF: [23.122600555419922, 46.245201110839844, 69.3677978515625, 92.49040222167969, 115.61299896240234, 138.735595703125, 161.8582000732422, 184.98080444335938]
模型加载完成。
CSV 读取完成，共 39544 个点。
裂变源：S_ref=1.190174e+00, S_pred=2.475779e+03, scale=S_ref/S_pred=4.807272e-04


C:\Users\Administrator\AppData\Local\Temp\ipykernel_37556\426934957.py:322: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



===== 归一化后误差指标（按区域）=====
   region     N group       L2_rel          MSE         RMSE          MAE    MaxRelErr   MeanRelErr           R2
     mat1  7460  phi1 3.652751e-02 2.048567e-09 4.526110e-05 3.796633e-05 6.774431e-02 2.768931e-02 7.683577e-01
     mat1  7460  phi2 3.684701e-02 1.098625e-10 1.048153e-05 8.728542e-06 7.906200e-02 2.767961e-02 8.106293e-01
     mat2  4371  phi1 1.683506e-02 5.221161e-10 2.284986e-05 2.064180e-05 3.071851e-02 1.383134e-02 8.867922e-01
     mat2  4371  phi2 1.691115e-02 2.498715e-11 4.998715e-06 4.475045e-06 3.401923e-02 1.373788e-02 9.217775e-01
     mat3  9675  phi1 5.199487e-02 6.904401e-11 8.309272e-06 5.721071e-06 8.249062e-02 2.056194e-02 9.952855e-01
     mat3  9675  phi2 5.255102e-02 6.037666e-12 2.457166e-06 1.740454e-06 1.531204e-01 2.548066e-02 9.947569e-01
     mat4  7346  phi1 4.532386e-02 1.694940e-09 4.116965e-05 3.566549e-05 6.872174e-02 3.440911e-02 9.808924e-01
     mat4  7346  phi2 4.545921e-02 6.150459e-11 7.842486e-06 6.774612e

C:\Users\Administrator\AppData\Local\Temp\ipykernel_37556\426934957.py:524: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
